# Labour market (counts) → ITL2

Builds **labour-market COUNTS per ITL2 region per year**: 16–64 population, economically active, employed, unemployed, economically inactive (APS), plus the Claimant Count as a cross-check. Saves `clean/labour_counts_itl2.csv`.

**Counts only, on purpose.** We SUM LA counts to ITL2 (exact aggregation) and derive rates *at* ITL2 later — never averaging LA-level rates:
- participation rate = active / pop_16_64
- inactivity rate = inactive / pop_16_64
- employment rate = employed / pop_16_64
- unemployment rate = unemployed / active

**How the APS counts are obtained (Nomis `NM_17_5`).** The APS dataset only holds *% variables*; the counts are their **Numerator** and **Denominator** measures (there is no "Value" measure). We use:

| variable | numerator | denominator |
|---|---|---|
| 18 Economic activity rate 16–64 | econ active 16–64 | pop 16–64 |
| 45 Employment rate 16–64 | employed 16–64 | pop 16–64 |

**Unemployed and inactive are DERIVED, not downloaded**: `unemployed = active − employed`, `inactive = pop − active`. These are exact APS identities (checked below, ±100 rounding). The published unemployment numerator (var 84) is **suppressed for 100–290 LAs a year** (small samples), so summing it would silently undercount ITL2 unemployment by ~⅓. Active/employed/pop are suppressed only for City of London / Orkney-type tiny LAs; those are reported in `aps_las_missing`.

**Why not pull ITL2 directly?** Nomis has an *ITL2 (Jan 2025)* geography, but it is only populated from 2025. We use it as a **validation target**: LA sums must reproduce it.

**Periods.** APS is a 12-month rolling series published quarterly (Jan–Dec, Apr–Mar, Jul–Jun, Oct–Sep). We keep **calendar-year (Jan–Dec) periods only** — mixing them would sum overlapping windows (~4× the true count).

**Northern Ireland (TLN0)** has no LA-level APS on Nomis; TLN0 = the whole of NI, so we take the NI country figure.

**⚠ Data-quality caveat (log this).** APS lost its accredited-official-statistics status from 2024 due to LFS response-rate problems; ONS is reweighting through 2025–26 and moving to a Transformed LFS. Recent years (2020+, acute 2023–24) are less reliable. The **Claimant Count** (administrative) is the cross-check — but note its own break: Universal Credit rollout from ~2013 progressively changed who is counted. Claimant count is age 16+ (APS is 16–64).

In [1]:
import io, urllib.request
from datetime import date
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
SPINE     = BASE / "clean" / "geography_spine.csv"
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
RAW_DIR   = BASE / "raw" / "labour"
OUT       = BASE / "clean" / "labour_counts_itl2.csv"
FIRST_YEAR = 2004                     # APS LA series starts Jan-Dec 2004

LA_TYPE   = "TYPE424"   # local authorities: district / unitary (as of April 2023)
ITL2_TYPE = "TYPE420"   # ITL2 (as of Jan 2025) - only populated from 2025; used for validation
NI_CODE, NI_ITL2 = "N92000002", "TLN0"

spine = pd.read_csv(SPINE)
lad2itl   = dict(zip(spine.la_code, spine.itl2_code))
itl2_name = dict(zip(spine.itl2_code, spine.itl2_name))
remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code","new_code")])) if CROSSWALK.exists() else {}
RAW_DIR.mkdir(parents=True, exist_ok=True)

def nomis(dataset, tag=None, **params):
    """Query the Nomis API, paging past its 25,000-row cap (a single request is
    silently truncated). Saves the raw CSV to raw/labour (dated) if tag is given."""
    q = "&".join(f"{k}={v}" for k, v in params.items())
    pages, offset = [], 0
    while True:
        url = f"https://www.nomisweb.co.uk/api/v01/dataset/{dataset}.data.csv?{q}&recordoffset={offset}"
        with urllib.request.urlopen(url, timeout=300) as r:
            page = pd.read_csv(io.BytesIO(r.read()), encoding="utf-8-sig")
        pages.append(page)
        if len(page) < 25000:
            break
        offset += 25000
    df = pd.concat(pages, ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    if tag:
        df.to_csv(RAW_DIR / f"nomis_{tag}_{date.today().isoformat()}.csv", index=False)
    return df

## 1. APS counts → ITL2 (sum over LAs)

Pull numerator + denominator for variables 18 and 45, keep Jan–Dec periods, derive unemployed / inactive at LA level, then sum to ITL2. An LA is used only if pop, active **and** employed are all present, so every ITL2 total is built from a consistent set of LAs.

In [2]:
APS_SEL = "date,date_name,geography_code,geography_name,variable,measures_name,obs_value,obs_status"

def aps_wide(raw):
    """Long Nomis APS extract -> one row per geography x calendar year with counts."""
    raw = raw[raw["DATE"].astype(str).str.endswith("-12")].copy()      # Jan-Dec periods only
    raw["year"] = raw["DATE"].astype(str).str[:4].astype(int)
    raw = raw[raw.year >= FIRST_YEAR]
    raw["key"] = raw["VARIABLE"].astype(str) + "_" + raw["MEASURES_NAME"].str.strip()
    w = raw.pivot_table(index=["GEOGRAPHY_CODE","year"], columns="key", values="OBS_VALUE", aggfunc="first")
    out = pd.DataFrame({
        "pop_16_64":         w["18_Denominator"],
        "econ_active_16_64": w["18_Numerator"],
        "employed_16_64":    w["45_Numerator"],
    })
    out["unemployed_16_64"]    = out.econ_active_16_64 - out.employed_16_64
    out["econ_inactive_16_64"] = out.pop_16_64 - out.econ_active_16_64
    return out.reset_index().rename(columns={"GEOGRAPHY_CODE":"geo"})

aps_raw = nomis("NM_17_5", tag="aps_la", geography=LA_TYPE, variable="18,45",
                measures="21001,21002", date="first-latest", select=APS_SEL)
la = aps_wide(aps_raw)
la["la_code"]   = la["geo"].astype(str).str.strip().replace(remap)
la["itl2_code"] = la["la_code"].map(lad2itl)

unmatched = sorted(la.loc[la.itl2_code.isna(), "la_code"].unique())
print("LA codes not in spine (dropped):", unmatched or "none")
la = la.dropna(subset=["itl2_code"])
assert not la.dropna(subset=["econ_active_16_64"]).duplicated(["la_code","year"]).any(), "old+new LA codes both populated -> double count"

COUNTS = ["pop_16_64","econ_active_16_64","employed_16_64","unemployed_16_64","econ_inactive_16_64"]
complete = la[["pop_16_64","econ_active_16_64","employed_16_64"]].notna().all(axis=1)
missing = la.loc[~complete].groupby(["itl2_code","year"])["la_code"].apply(lambda s: ",".join(sorted(s)))
print("LA-years dropped for suppression:", (~complete).sum(), "->", sorted(la.loc[~complete, "la_code"].unique()))

aps = la[complete].groupby(["itl2_code","year"])[COUNTS].sum().reset_index()
aps["aps_las_missing"] = aps.set_index(["itl2_code","year"]).index.map(missing).fillna("")

# Northern Ireland: no LA-level APS on Nomis; TLN0 == NI country
ni_raw = nomis("NM_17_5", tag="aps_ni", geography="TYPE499", variable="18,45",
               measures="21001,21002", date="first-latest", select=APS_SEL)
ni = aps_wide(ni_raw[ni_raw.GEOGRAPHY_CODE == NI_CODE]).drop(columns="geo").assign(itl2_code=NI_ITL2, aps_las_missing="")
aps = pd.concat([aps, ni], ignore_index=True)

print(f"APS: {aps.itl2_code.nunique()} regions x {aps.year.nunique()} years ({aps.year.min()}-{aps.year.max()})")
aps.head()

LA codes not in spine (dropped): none
LA-years dropped for suppression: 2 -> ['E09000001', 'S12000023']


APS: 46 regions x 22 years (2004-2025)


,itl2_code,year,pop_16_64,econ_active_16_64,employed_16_64,unemployed_16_64,econ_inactive_16_64,aps_las_missing
0,TLC3,2004,412700.0,300800.0,281800.0,19000.0,111900.0,
1,TLC3,2005,413900.0,301500.0,284100.0,17400.0,112400.0,
2,TLC3,2006,416700.0,305600.0,286900.0,18700.0,111100.0,
3,TLC3,2007,418800.0,307000.0,285200.0,21800.0,111800.0,
4,TLC3,2008,419900.0,304300.0,278800.0,25500.0,115600.0,


### 1b. Validation

(i) The derivation identities hold where the published numerators exist; (ii) for 2025, our LA sums reproduce Nomis's direct ITL2 estimates. Gaps should be ~0% (rounding only) except where an LA was suppressed.

In [3]:
# (i) identities, on LAs where var 84 / 111 numerators are published
chk = nomis("NM_17_5", geography=LA_TYPE, variable="84,111", measures="21001",
            date=f"{la.year.max()}-12", select="geography_code,variable,obs_value")
chk = chk.pivot_table(index="GEOGRAPHY_CODE", columns="VARIABLE", values="OBS_VALUE")
latest_la = la[la.year == la.year.max()].set_index("geo")
j = latest_la.join(chk).dropna()
print(f"Identity check, {la.year.max()} ({len(j)} LAs): max |derived unemployed - published| = "
      f"{(j.unemployed_16_64 - j[84]).abs().max():.0f};  max |derived inactive - published| = "
      f"{(j.econ_inactive_16_64 - j[111]).abs().max():.0f}   (expect <= ~100, rounding)")

# (ii) LA sums vs direct ITL2 (only populated from 2025)
direct = aps_wide(nomis("NM_17_5", geography=ITL2_TYPE, variable="18,45", measures="21001,21002",
                        date="first-latest", select=APS_SEL)).dropna(subset=["pop_16_64"])
cmp = aps.merge(direct, left_on=["itl2_code","year"], right_on=["geo","year"], suffixes=("","_direct"))
for v in ["pop_16_64","econ_active_16_64","unemployed_16_64"]:
    gap = 100 * (cmp[v] / cmp[f"{v}_direct"] - 1)
    worst = cmp.loc[gap.abs().idxmax()]
    print(f"{v:18s} LA-sum vs direct ITL2 ({len(cmp)} region-years): median {gap.median():+.2f}%, "
          f"worst {worst.itl2_code} {worst.year} {gap.abs().max():.1f}%  {worst.aps_las_missing}")

Identity check, 2025 (132 LAs): max |derived unemployed - published| = 100;  max |derived inactive - published| = 100   (expect <= ~100, rounding)


pop_16_64          LA-sum vs direct ITL2 (45 region-years): median +0.00%, worst TLI3 2025 0.1%  
econ_active_16_64  LA-sum vs direct ITL2 (45 region-years): median +0.00%, worst TLI3 2025 0.2%  
unemployed_16_64   LA-sum vs direct ITL2 (45 region-years): median +0.00%, worst TLD1 2025 1.7%  


## 2. Claimant Count → ITL2 (sum over LAs, average over months)

Nomis `NM_162_1` (all ages 16+, both sexes, count). The claimant count is a monthly *stock*: it sums across LAs but is **averaged** across months to an annual level. Only complete years (12 months) are kept.

In [4]:
cc = nomis("NM_162_1", tag="claimant_la", geography=LA_TYPE, gender="0", age="0", measure="1",
           measures="20100", date="first-latest", select="date,geography_code,obs_value")
cc["year"] = cc["DATE"].astype(str).str[:4].astype(int)
cc = cc[cc.year >= FIRST_YEAR]
cc["la_code"]   = cc["GEOGRAPHY_CODE"].astype(str).str.strip().replace(remap)
cc["itl2_code"] = cc["la_code"].map(lad2itl)
print("Claimant LA codes not in spine:", sorted(cc.loc[cc.itl2_code.isna(), "la_code"].unique()) or "none")
cc = cc.dropna(subset=["itl2_code", "OBS_VALUE"])

by_month = cc.groupby(["itl2_code","year","DATE"])["OBS_VALUE"].sum().reset_index()
claim = (by_month.groupby(["itl2_code","year"])
                 .agg(claimant_count=("OBS_VALUE","mean"), months=("DATE","nunique"))
                 .reset_index())
claim = claim[claim.months == 12].drop(columns="months")
print(f"Claimant count: {claim.itl2_code.nunique()} regions x {claim.year.nunique()} years "
      f"({claim.year.min()}-{claim.year.max()})")
claim.head()

Claimant LA codes not in spine: none
Claimant count: 46 regions x 22 years (2004-2025)


,itl2_code,year,claimant_count
0,TLC3,2004,14667.916667
1,TLC3,2005,13872.916667
2,TLC3,2006,15360.833333
3,TLC3,2007,15219.166667
4,TLC3,2008,16349.166667


## 3. Combine and save (counts only)

Rates are **not** computed here — derive them later from these columns.

In [5]:
panel = aps.merge(claim, on=["itl2_code","year"], how="outer")
panel["itl2_name"] = panel["itl2_code"].map(itl2_name)
front = ["itl2_code","itl2_name","year"]
panel = panel[front + COUNTS + ["claimant_count","aps_las_missing"]].sort_values(["year","itl2_code"]).reset_index(drop=True)
OUT.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUT, index=False)
print(f"Saved -> {OUT}")
print("Columns:", list(panel.columns))
panel.head()

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/labour_counts_itl2.csv
Columns: ['itl2_code', 'itl2_name', 'year', 'pop_16_64', 'econ_active_16_64', 'employed_16_64', 'unemployed_16_64', 'econ_inactive_16_64', 'claimant_count', 'aps_las_missing']


,itl2_code,itl2_name,year,pop_16_64,econ_active_16_64,employed_16_64,unemployed_16_64,econ_inactive_16_64,claimant_count,aps_las_missing
0,TLC3,Tees Valley,2004,412700.0,300800.0,281800.0,19000.0,111900.0,14667.916667,
1,TLC4,"Northumberland, Durham and Tyne & Wear",2004,1208600.0,872200.0,822200.0,50000.0,336400.0,32422.500000,
2,TLD1,Cumbria,2004,309100.0,235300.0,226800.0,8500.0,73800.0,5583.750000,
3,TLD3,Greater Manchester,2004,1648700.0,1219100.0,1159400.0,59700.0,429600.0,38301.666667,
4,TLD4,Lancashire,2004,900800.0,673400.0,642000.0,31400.0,227400.0,16659.166667,


## 4. Coverage & sanity check

Expect 46 ITL2 regions. A derived unemployment rate (unemployed/active) should look plausible (a few % up to ~8%), and UK totals should be ~32–33m active.

In [6]:
regions = spine[["itl2_code","itl2_name"]].drop_duplicates()
a = panel.dropna(subset=["econ_active_16_64"])
print(f"APS coverage: {a.itl2_code.nunique()} of {len(regions)} regions x {a.year.nunique()} years ({a.year.min()}-{a.year.max()})")
print("Regions with no APS data:", sorted(set(regions.itl2_code) - set(a.itl2_code)) or "none")

uk = a.groupby("year")[COUNTS].sum() / 1e6
print("\nUK totals (millions):"); print(uk.loc[[uk.index.min(), 2019, uk.index.max()]].round(2).to_string())

chk = a.copy()
chk["unemp_rate_%"] = 100 * chk["unemployed_16_64"] / chk["econ_active_16_64"]
latest = chk.year.max()
print(f"\nDerived unemployment rate, {latest} (highest 3 / lowest 3):")
print(pd.concat([chk[chk.year==latest].nlargest(3,"unemp_rate_%"), chk[chk.year==latest].nsmallest(3,"unemp_rate_%")])
        [["itl2_name","unemp_rate_%"]].round(1).to_string(index=False))

APS coverage: 46 of 46 regions x 22 years (2004-2025)
Regions with no APS data: none

UK totals (millions):
      pop_16_64  econ_active_16_64  employed_16_64  unemployed_16_64  econ_inactive_16_64
year                                                                                     
2004      38.34              29.18           27.77              1.41                 9.15
2019      41.32              32.55           31.26              1.29                 8.77
2025      42.03              33.23           31.73              1.50                 8.80

Derived unemployment rate, 2025 (highest 3 / lowest 3):
                         itl2_name  unemp_rate_%
                     West Midlands           7.6
Outer London - West and North West           7.1
               Inner London - East           6.9
                   North Yorkshire           1.7
             East Central Scotland           2.0
                  Northern Ireland           2.1
